In [5]:
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

def proyecto_mineria_datos_completo():
    """Proyecto completo: Detective de Datos - Supermercado ABC (Corregido)"""
    print("="*60)
    print(" PROYECTO DE MINERÍA DE DATOS: SUPERMERCADO ABC")
    print("="*60 + "\n")

    # =========================================================================
    # ACTIVIDAD 1: Análisis de Canasta de Compra (Reglas de Asociación)
    # =========================================================================
    print("--- [Actividad 1] Análisis de Canasta de Compra ---")
    np.random.seed(42)
    n_transacciones = 5000

    patrones_conocidos = [
        {'antecedentes': ['Pan', 'Leche'], 'consecuentes': ['Mantequilla']},
        {'antecedentes': ['Cerveza', 'Papas'], 'consecuentes': ['Dulces']},
        {'antecedentes': ['Carne', 'Verduras'], 'consecuentes': ['Salsa']}
    ]

    print("Patrones objetivo esperados:")
    for i, patron in enumerate(patrones_conocidos):
        print(f"  {i+1}. {patron['antecedentes']} → {patron['consecuentes']}")

    # Simulación de transacciones 
    transacciones = []
    productos_extras = ['Manzanas', 'Refresco', 'Arroz', 'Frijoles', 'Yogur']

    for _ in range(n_transacciones):
        items_transaccion = set()
        if np.random.rand() > 0.3:
            items_transaccion.add(np.random.choice(productos_extras))
            
        rand_val = np.random.rand()
        if rand_val < 0.25:
            items_transaccion.update(['Pan', 'Leche', 'Mantequilla'])
        elif 0.25 <= rand_val < 0.45:
            items_transaccion.update(['Cerveza', 'Papas', 'Dulces'])
        elif 0.45 <= rand_val < 0.60:
            items_transaccion.update(['Carne', 'Verduras', 'Salsa'])
            
        if len(items_transaccion) > 0:
            transacciones.append(list(items_transaccion))

    # Aplicar Apriori para encontrar conjuntos frecuentes
    te = TransactionEncoder()
    df_trans = pd.DataFrame(te.fit(transacciones).transform(transacciones), columns=te.columns_)
    itemsets_frecuentes = apriori(df_trans, min_support=0.04, use_colnames=True)

    # Cálculo de reglas de asociación usando tuplas (para evitar el error de hasheo)
    itemsets_frecuentes['itemsets'] = itemsets_frecuentes['itemsets'].apply(lambda x: set(x))
    itemsets_frecuentes['length'] = itemsets_frecuentes['itemsets'].apply(lambda x: len(x))
    
    reglas_list = []
    for idx, row in itemsets_frecuentes[itemsets_frecuentes['length'] > 1].iterrows():
        itemset = row['itemsets']
        support_xy = row['support']
        
        for item in itemset:
            antecedente = itemset - {item}
            consecuente = {item}
            
            match_ant = itemsets_frecuentes[itemsets_frecuentes['itemsets'] == antecedente]
            if not match_ant.empty:
                support_ant = match_ant['support'].values[0]
                confidence = support_xy / support_ant
                if confidence >= 0.3:  # Umbral de confianza mínimo
                    reglas_list.append({
                        'antecedents': tuple(sorted(antecedente)),  # Usamos tuplas para ser hasheable
                        'consecuentes': tuple(sorted(consecuente)),  # Usamos tuplas para ser hasheable
                        'support': support_xy,
                        'confidence': confidence
                    })

    df_reglas = pd.DataFrame(reglas_list).drop_duplicates().sort_values(by='confidence', ascending=False)

    print("\nReglas de asociación descubiertas (Top 5 por confianza):")
    if not df_reglas.empty:
        print(df_reglas[['antecedents', 'consecuentes', 'support', 'confidence']]
              .head(5)
              .to_string(index=False))
    else:
        print("No se encontraron reglas con los umbrales actuales.")
    print("\n" + "-"*60 + "\n")

    # =========================================================================
    # ACTIVIDAD 2: Segmentación de Clientes (K-Means)
    # =========================================================================
    print("--- [Actividad 2] Segmentación de Clientes ---")
    n_clientes = 1000
    
    clientes_data = {
        'cliente_id': range(1, n_clientes + 1),
        'frecuencia_visitas': np.random.poisson(lam=5, size=n_clientes),
        'gasto_total': np.random.normal(loc=1500, scale=400, size=n_clientes),
        'promedio_items': np.random.normal(loc=6, scale=2, size=n_clientes)
    }
    df_clientes = pd.DataFrame(clientes_data)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_clientes[['frecuencia_visitas', 'gasto_total', 'promedio_items']])

    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    df_clientes['segmento'] = kmeans.fit_predict(X_scaled)

    print("Promedios por Segmento de Cliente identificado:")
    resumen_segmentos = df_clientes.groupby('segmento')[['frecuencia_visitas', 'gasto_total', 'promedio_items']].mean()
    print(resumen_segmentos.round(2))
    print("\n" + "-"*60 + "\n")

    # =========================================================================
    # ACTIVIDAD 3: Detección de Anomalías 
    # =========================================================================
    print("--- [Actividad 3] Detección de Anomalías y Fraudes ---")
    n_pagos = 4900
    
    montos_normales = np.random.normal(loc=50.0, scale=15.0, size=n_pagos)
    items_normales = np.random.poisson(lam=4, size=n_pagos)
    
    n_anomalias = 100
    montos_anomalos = np.random.uniform(low=600.0, high=2500.0, size=n_anomalias)
    items_anomalos = np.random.randint(low=30, high=60, size=n_anomalias)
    
    montos = np.concatenate([montos_normales, montos_anomalos])
    items_cant = np.concatenate([items_normales, items_anomalos])
    
    df_transacciones_fraude = pd.DataFrame({
        'monto_total': montos,
        'cantidad_items': items_cant
    })

    iso = IsolationForest(contamination=0.02, random_state=42)
    df_transacciones_fraude['estado'] = iso.fit_predict(df_transacciones_fraude[['monto_total', 'cantidad_items']])
    
    anomalias_detectadas = (df_transacciones_fraude['estado'] == -1).sum()
    
    print(f"Total de transacciones evaluadas: {len(df_transacciones_fraude)}")
    print(f"Transacciones sospechosas detectadas por el modelo: {anomalias_detectadas}")
    print("\nEjemplos de transacciones detectadas como anomalías (posible fraude):")
    print(df_transacciones_fraude[df_transacciones_fraude['estado'] == -1].head())

    print("\n" + "="*60)
    print("="*60)

    return df_trans, df_clientes, df_transacciones_fraude

# Ejecutar el proyecto completo
if __name__ == "__main__":
    resultados = proyecto_mineria_datos_completo()

 PROYECTO DE MINERÍA DE DATOS: SUPERMERCADO ABC

--- [Actividad 1] Análisis de Canasta de Compra ---
Patrones objetivo esperados:
  1. ['Pan', 'Leche'] → ['Mantequilla']
  2. ['Cerveza', 'Papas'] → ['Dulces']
  3. ['Carne', 'Verduras'] → ['Salsa']

Reglas de asociación descubiertas (Top 5 por confianza):
            antecedents   consecuentes  support  confidence
               (Carne,)       (Salsa,) 0.170198         1.0
        (Manzanas, Pan) (Mantequilla,) 0.043759         1.0
(Mantequilla, Refresco)       (Leche,) 0.040304         1.0
      (Leche, Refresco) (Mantequilla,) 0.040304         1.0
   (Mantequilla, Yogur)       (Leche,) 0.040304         1.0

------------------------------------------------------------

--- [Actividad 2] Segmentación de Clientes ---
Promedios por Segmento de Cliente identificado:
          frecuencia_visitas  gasto_total  promedio_items
segmento                                                 
0                       3.85      1262.66            7.20
1 